In [0]:
# Text widgets

dbutils.widgets.text(
    "catalog",
    "workspace",
    "Catalog"
)

dbutils.widgets.text(
    "silver_schema",
    "nyc_taxi_silver",
    "Silver Schema"
)

dbutils.widgets.text(
    "gold_schema",
    "nyc_taxi_gold",
    "Gold Schema"
)

dbutils.widgets.text(
    "start_date",
    "2024-12-01",
    "Start Date (YYYY-MM-DD)"
)

dbutils.widgets.text(
    "end_date",
    "2025-02-28",
    "End Date (YYYY-MM-DD)"
)

In [0]:
catalog = dbutils.widgets.get("catalog")

silver_schema = dbutils.widgets.get("silver_schema")

gold_schema = dbutils.widgets.get("gold_schema")

start_date = dbutils.widgets.get("start_date")

end_date = dbutils.widgets.get("end_date")

print(catalog)
print(silver_schema)
print(gold_schema)
print(start_date)
print(end_date)
trips_df = spark.table(
    f"{catalog}.{silver_schema}.unified_trips"
)

zones_df = spark.table(
    f"{catalog}.{silver_schema}.dim_taxi_zones"
)

In [0]:
import sys
import importlib

PROJECT_ROOT = (
    "/Workspace/Users/anmolmishra02@gmail.com/"
    "NYC-Trip-Records-Pipeline/take_two_nyc_taxi"
)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import src.gold.aggregations as aggregations

importlib.reload(aggregations)

build_daily_revenue = aggregations.build_daily_revenue
daily_revenue_df = build_daily_revenue(
    trips_df,
    zones_df,
    start_date=start_date,
    end_date=end_date
)

display(daily_revenue_df.limit(10))

In [0]:
daily_revenue_table = (
    f"{catalog}.{gold_schema}.daily_revenue"
)

(
    daily_revenue_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(daily_revenue_table)
)

print(f"Successfully created: {daily_revenue_table}")

In [0]:
%sql
SELECT
    service_type,
    SUM(trip_count) AS total_trips,
    SUM(collected_trip_count) AS collected_trips,
    SUM(collected_revenue) AS total_revenue,
    SUM(excluded_payment_trip_count) AS excluded_trips
FROM workspace.nyc_taxi_gold.daily_revenue
GROUP BY service_type;

In [0]:
%sql
SELECT
    service_type,
    COUNT(*) AS total_silver_records,
    SUM(
        CASE
            WHEN TO_DATE(pickup_datetime)
                BETWEEN '2024-12-01' AND '2025-02-28'
            THEN 1 ELSE 0
        END
    ) AS records_in_date_range,
    SUM(
        CASE
            WHEN TO_DATE(pickup_datetime) < '2024-12-01'
              OR TO_DATE(pickup_datetime) > '2025-02-28'
            THEN 1 ELSE 0
        END
    ) AS records_outside_date_range
FROM workspace.nyc_taxi_silver.unified_trips
GROUP BY service_type;

In [0]:
%sql
SELECT
    location_id,
    borough,
    zone,
    service_zone
FROM workspace.nyc_taxi_silver.dim_taxi_zones
WHERE LOWER(zone) LIKE '%airport%'
   OR LOWER(zone) LIKE '%newark%'
ORDER BY zone;

In [0]:
importlib.reload(aggregations)

build_airport_analysis = aggregations.build_airport_analysis
airport_analysis_df = build_airport_analysis(
    trips_df=trips_df,
    zones_df=zones_df,
    start_date=start_date,
    end_date=end_date
)

display(airport_analysis_df.orderBy(
    "month",
    "airport",
    "service_type",
    "direction"
))
airport_table = f"{catalog}.{gold_schema}.airport_analysis"

(
    airport_analysis_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(airport_table)
)

print(f"Created: {airport_table}")

In [0]:
%sql
SELECT
    distinct month
    
FROM workspace.nyc_taxi_gold.airport_analysis
;

In [0]:
%sql
SELECT
    month,
    airport,
    service_type,
    direction,
    trip_count,
    ROUND(average_fare, 2) AS average_fare,
    credit_card_trip_count,
    observed_card_tip_count,
    tipped_card_trip_count,
    card_tip_rate_pct,
    ROUND(average_card_tip, 2) AS average_card_tip
FROM workspace.nyc_taxi_gold.airport_analysis
ORDER BY month, airport, service_type, direction;

In [0]:
%sql
SELECT
    source_month,
    service_type,
    pickup_datetime,
    dropoff_datetime,
    pickup_location_id,
    dropoff_location_id
FROM workspace.nyc_taxi_silver.unified_trips
WHERE
    YEAR(pickup_datetime) < 2024
    OR pickup_datetime < TIMESTAMP('2024-12-01')
    OR pickup_datetime >= TIMESTAMP('2025-03-01')
    OR YEAR(dropoff_datetime) < 2024
    OR dropoff_datetime < TIMESTAMP('2024-12-01')
    OR dropoff_datetime >= TIMESTAMP('2025-03-01')


In [0]:
%sql
SELECT
    source_month,
    service_type,
    COUNT(*) AS total_flagged,

    SUM(CASE WHEN pickup_datetime < TIMESTAMP('2024-12-01')
             THEN 1 ELSE 0 END) AS pickup_before_window,

    SUM(CASE WHEN pickup_datetime >= TIMESTAMP('2025-03-01')
             THEN 1 ELSE 0 END) AS pickup_after_window,

    SUM(CASE WHEN dropoff_datetime < TIMESTAMP('2024-12-01')
             THEN 1 ELSE 0 END) AS dropoff_before_window,

    SUM(CASE WHEN dropoff_datetime >= TIMESTAMP('2025-03-01')
             THEN 1 ELSE 0 END) AS dropoff_after_window,

    SUM(CASE WHEN YEAR(pickup_datetime) < 2020
             OR YEAR(dropoff_datetime) < 2020
             THEN 1 ELSE 0 END) AS suspicious_old_timestamps

FROM workspace.nyc_taxi_silver.unified_trips

WHERE
    pickup_datetime < TIMESTAMP('2024-12-01')
    OR pickup_datetime >= TIMESTAMP('2025-03-01')
    OR dropoff_datetime < TIMESTAMP('2024-12-01')
    OR dropoff_datetime >= TIMESTAMP('2025-03-01')

GROUP BY source_month, service_type
ORDER BY source_month, service_type;

In [0]:
%sql
SELECT
    source_month,
    service_type,
    source_file,
    pickup_datetime,
    dropoff_datetime,
    pickup_location_id,
    dropoff_location_id,
    fare_amount,
    total_amount
FROM workspace.nyc_taxi_silver.unified_trips
WHERE
    YEAR(pickup_datetime) < 2020
    OR YEAR(dropoff_datetime) < 2020
ORDER BY source_month, service_type;

In [0]:
%sql
SELECT
    location_id,
    zone,
    service_zone
FROM workspace.nyc_taxi_silver.dim_taxi_zones
WHERE LOWER(borough) = 'manhattan'
ORDER BY location_id;

In [0]:
%sql
SELECT
    service_type,
    CASE
        WHEN pickup_location_id NOT IN (
            4,12,13,45,48,50,68,79,87,88,90,100,107,
            113,114,125,137,144,148,158,161,162,163,
            164,170,186,209,211,229,230,231,232,233,
            234,246,249,261
        )
        AND dropoff_location_id IN (
            4,12,13,45,48,50,68,79,87,88,90,100,107,
            113,114,125,137,144,148,158,161,162,163,
            164,170,186,209,211,229,230,231,232,233,
            234,246,249,261
        )
        THEN 'INBOUND_CBD'

        WHEN pickup_location_id IN (
            4,12,13,45,48,50,68,79,87,88,90,100,107,
            113,114,125,137,144,148,158,161,162,163,
            164,170,186,209,211,229,230,231,232,233,
            234,246,249,261
        )
        AND dropoff_location_id NOT IN (
            4,12,13,45,48,50,68,79,87,88,90,100,107,
            113,114,125,137,144,148,158,161,162,163,
            164,170,186,209,211,229,230,231,232,233,
            234,246,249,261
        )
        THEN 'OUTBOUND_CBD'

        WHEN pickup_location_id IN (
            4,12,13,45,48,50,68,79,87,88,90,100,107,
            113,114,125,137,144,148,158,161,162,163,
            164,170,186,209,211,229,230,231,232,233,
            234,246,249,261
        )
        AND dropoff_location_id IN (
            4,12,13,45,48,50,68,79,87,88,90,100,107,
            113,114,125,137,144,148,158,161,162,163,
            164,170,186,209,211,229,230,231,232,233,
            234,246,249,261
        )
        THEN 'WITHIN_CBD'

        ELSE 'OUTSIDE_CBD'
    END AS trip_category,
    COUNT(*) AS trip_count
FROM workspace.nyc_taxi_silver.unified_trips
GROUP BY service_type, trip_category
ORDER BY service_type, trip_category;

In [0]:
import importlib
import src.gold.aggregations as aggregations
# Set timezone to New York
spark.conf.set("spark.sql.session.timeZone", "America/New_York")

# Verify
print(spark.conf.get("spark.sql.session.timeZone"))
importlib.reload(aggregations)

congestion_df = aggregations.build_congestion_pricing_impact(
    trips_df,
    start_date,
    end_date
)

display(congestion_df)

In [0]:
congestion_table = f"{catalog}.{gold_schema}.congestion_pricing_impact"

(
    congestion_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(congestion_table)
)

In [0]:
%sql

SELECT *
FROM workspace.nyc_taxi_gold.congestion_pricing_impact
ORDER BY service_type, period;


In [0]:
%sql
SELECT
    period,
    service_type,
    COUNT(*) AS negative_fee_records,
    SUM(cbd_congestion_fee) AS negative_fee_total
FROM (
    SELECT
        CASE
            WHEN dropoff_datetime <
                CAST('2025-01-05 00:00:00' AS TIMESTAMP_NTZ)
            THEN 'BEFORE'
            ELSE 'AFTER'
        END AS period,
        service_type,
        cbd_congestion_fee
    FROM workspace.nyc_taxi_silver.unified_trips
    WHERE pickup_location_id NOT IN (
        4,12,13,45,48,50,68,79,87,88,90,100,107,113,114,
        125,137,144,148,158,161,162,163,164,170,186,209,
        211,229,230,231,232,233,234,246,249,261
    )
    AND dropoff_location_id IN (
        4,12,13,45,48,50,68,79,87,88,90,100,107,113,114,
        125,137,144,148,158,161,162,163,164,170,186,209,
        211,229,230,231,232,233,234,246,249,261
    )
    AND dropoff_datetime >= CAST('2024-12-01 00:00:00' AS TIMESTAMP_NTZ)
    AND dropoff_datetime < CAST('2025-03-01 00:00:00' AS TIMESTAMP_NTZ)
    AND cbd_congestion_fee < 0
)
GROUP BY period, service_type
ORDER BY period, service_type;

In [0]:
%sql
SELECT
    source_month,
    service_type,
    pickup_datetime,
    dropoff_datetime,
    pickup_location_id,
    dropoff_location_id,
    payment_type,
    total_amount,
    cbd_congestion_fee
FROM workspace.nyc_taxi_silver.unified_trips
WHERE
    dropoff_datetime < TIMESTAMP('2025-01-05')
    AND cbd_congestion_fee > 0
ORDER BY dropoff_datetime;

In [0]:
%sql
SELECT
    pickup_location_id,
    dropoff_location_id,
    pickup_datetime,
    dropoff_datetime,
    cbd_congestion_fee,
    CASE
        WHEN pickup_location_id NOT IN (
            4,12,13,45,48,50,68,79,87,88,90,100,107,
            113,114,125,137,144,148,158,161,162,163,
            164,170,186,209,211,229,230,231,232,233,
            234,246,249,261
        )
        AND dropoff_location_id IN (
            4,12,13,45,48,50,68,79,87,88,90,100,107,
            113,114,125,137,144,148,158,161,162,163,
            164,170,186,209,211,229,230,231,232,233,
            234,246,249,261
        )
        THEN 'INBOUND_CBD'
        ELSE 'OTHER'
    END AS trip_category
FROM workspace.nyc_taxi_silver.unified_trips
WHERE source_month = '2025-01'
  AND service_type = 'yellow'
  AND TO_DATE(dropoff_datetime) < DATE('2025-01-05')
  AND cbd_congestion_fee > 0;

In [0]:
%sql
SELECT
    payment_type,
    fare_amount,
    total_amount,
    cbd_congestion_fee,
    COUNT(*) AS trip_count
FROM workspace.nyc_taxi_silver.unified_trips
WHERE service_type = 'yellow'
  AND dropoff_datetime >= TIMESTAMP('2025-01-05 00:00:00')
  AND dropoff_datetime < TIMESTAMP('2025-03-01 00:00:00')
  AND cbd_congestion_fee < 0
  AND fare_amount >= 0
GROUP BY
    payment_type,
    fare_amount,
    total_amount,
    cbd_congestion_fee
ORDER BY trip_count DESC;